# 01 · Define & Explore — symmetric assembly concepts + the design plumbing

**Standard slot:** *define & explore.* **For Project 04 this means:** pin down the symmetry
concepts (symmetric contigs, tied positions, oligomeric-state error) and the three metrics that
judge a symmetric design, then stand up `sym_tools` and run a small **C3** end-to-end as your
"hello-world" (D0).

Run `00_setup.ipynb` first in this session.

## The concepts, precisely

- **Symmetric contig** — the per-subunit shape spec given to RFdiffusion symmetric mode. You
  describe ONE subunit (the asymmetric unit); the point group (`C3`/`C4`/`D2`) replicates it.
- **Tied positions** — in ProteinMPNN, residues forced to decode to the SAME amino acid across
  all symmetry-related subunits. Tying makes the assembly symmetric in *sequence*, not just
  backbone — without it, a 'symmetric' design is a fiction.
- **Oligomeric-state error (wrong-oligomer)** — the central failure mode: a subunit designed for
  C3 may instead prefer a dimer or tetramer. The in-silico tell is when the sequence *also*
  scores well as the wrong order (we model that explicitly in notebook 04).

## The metrics, precisely

| Metric | Range | Means | Does **not** mean |
|--------|-------|-------|-------------------|
| subunit scRMSD | Å | designed-vs-predicted Cα-RMSD of **one subunit** | that the assembly forms |
| interface pAE | Å | inter-chain predicted aligned error at the interface (< 10 good) | the *right* oligomer / a guarantee |
| symmetry RMSD | Å | does the predicted assembly **close** into the target point group? | stability / binding |
| pLDDT | 0–100 | per-residue *local* confidence | thermostability / assembly |
| interface energy | REU | interface-energy proxy (more negative = better) | a hard pass/fail |

Write your own one-paragraph definitions in `D0` — including the 'does not mean' column. The
single most important honesty point: **interface pAE is necessary, not sufficient** — only nsEM /
SEC-MALS / native-MS confirm the actual oligomeric state.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Stand up `sym_tools`

`scripts/sym_tools.py` exposes three functions: `generate_symmetric(...)`,
`tied_mpnn(...)`, and `multimer_predict(...)`. The real backends (symmetric RFdiffusion,
tied ProteinMPNN, AF2-Multimer) need GPU + heavy installs and an **A100** for a full campaign
(see `MANUAL.md` §2); a deterministic **mock** backend lets you build and test the plumbing
first. **Never report mock numbers as real** — they are SYNTHETIC (`EXAMPLE_DATA`).

In [ ]:
from sym_tools import (generate_symmetric, tied_mpnn, multimer_predict,
                       symmetric_contig, tied_positions, SYMMETRY_ORDER)

print("symmetries this project supports (order = # subunits):")
for s in ["C3", "C4", "D2"]:
    print(f"  {s}: {SYMMETRY_ORDER[s]} subunits   example contig = {symmetric_contig(s, 60)!r}")

### Tied positions — what symmetry actually means in the sequence

`tied_positions(subunit_length, n_subunits)` returns, for each residue of the asymmetric unit,
the global indices across all chains that must share one amino acid. This is the bookkeeping
ProteinMPNN needs to design a genuinely symmetric sequence.

In [ ]:
groups = tied_positions(subunit_length=6, n_subunits=3)  # tiny example for readability
print("residue -> tied chain copies (global indices), C3 with a 6-residue subunit:")
for i, g in enumerate(groups):
    print(f"  subunit residue {i} ties global indices {g}")

## Hello-world: a small C3 (mock)

Generate one small **C3** backbone, design a tied sequence for it, and predict the assembly —
the whole pipeline in miniature. We use the **mock** backend so it runs anywhere; switch to the
real backends on an A100 (see the note below the cell).

In [ ]:
# Start with the mock backend to confirm the plumbing, THEN switch tool="rfdiffusion"/"proteinmpnn"/"af2".
asm = generate_symmetric("C3", length=60, n=1, tool="mock")[0]
print("backbone:", asm.as_row())

seqs = tied_mpnn(asm, "C3", n_seqs=2, tool="mock")
print("\ntied sequences (both subunits share each sequence):")
for s in seqs:
    print(" ", s)

score = multimer_predict(seqs[0], "C3", tool="mock")
print("\nAF2-Multimer readout (SYNTHETIC — mock):")
print(" ", score.as_row())

### Switch to the real backends

Once `00_setup` has installed symmetric RFdiffusion / ProteinMPNN / ColabFold this session
(A100 recommended), change `tool="mock"` to the real tool names. ESMFold-style triage is not
enough here — assembly judgement needs the **multimer** predictor. Record runtime + the
RFdiffusion/ColabFold commit pins in `LOG.md`.

In [ ]:
# Uncomment on an A100 once the real tools are installed in 00_setup:
# asm = generate_symmetric("C3", length=60, n=1, tool="rfdiffusion")[0]
# seqs = tied_mpnn(asm, "C3", n_seqs=2, tool="proteinmpnn")
# score = multimer_predict(seqs[0], "C3", tool="af2")
# assert score.ok, score.error
print("Ready — flip the tools to the real backends on an A100 (see MANUAL §2).")

## Visualize a structure (py3Dmol)

Use this to eyeball any predicted assembly PDB once you have one (color by chain to see the
symmetry). On the mock backend there is no PDB yet — this is the placeholder you'll fill on
Colab.

In [ ]:
import py3Dmol

def show_assembly(pdb_path_or_str, is_path=True):
    """Show a multi-chain assembly colored by chain (so symmetry is visible)."""
    data = open(pdb_path_or_str).read() if is_path else pdb_path_or_str
    view = py3Dmol.view(width=520, height=420)
    view.addModel(data, "pdb")
    view.setStyle({"cartoon": {"colorscheme": "chain"}})
    view.zoomTo()
    return view.show()

# Example (after a real multimer prediction):
# show_assembly(score_pdb_path)
print("show_assembly(pdb_path) ready — color-by-chain reveals the point group.")

## D0 checklist
- [ ] One-paragraph definition of each metric **with** its 'does not mean' note (esp. interface pAE = necessary-not-sufficient).
- [ ] `sym_tools` imported; tied-position layout understood for your target symmetry.
- [ ] One reproduced small-C3 run (mock, then real on an A100 if available) with its three metrics printed.
- [ ] Problem statement with measurable success criteria + controls.
- [ ] `LOG.md` entry: tool version / commit pins, GPU, runtime, seed.

**Next:** `02_generate.ipynb` — the C3/C4/D2 design campaign.